# 4주차 A1 · UCI HAR ResNet 1D (단독 효과)

A0 기준 CNN의 Conv 두 층을 **residual block 두 개**로 바꿉니다. 나머지(분할·정규화·GAP 헤드·학습 조건·평가)는 A0와 같습니다.

| 부분 | A0 | **A1** |
|---|---|---|
| 1번째 특징 블록 | Conv(32, k3) + BN | **Residual block 32** (Conv k3 → BN → ReLU → Conv k3 → BN, + shortcut) |
| 2번째 특징 블록 | Conv(64, k3) + BN | **Residual block 64** (같은 구조, 채널이 달라 1×1 projection shortcut) |
| 블록 뒤 | MaxPooling(2) | MaxPooling(2) (동일) |
| 분류 헤드 | GAP → Dropout 0.4 → Dense 6 | 동일 |

확인할 질문: **shortcut(`Add`)이 기준 CNN보다 학습·일반화를 돕는가?**

residual block은 Conv가 블록마다 2개라서 A0보다 깊어집니다. 그래서 `USE_RESIDUAL=False`(shortcut만 뺀 같은 깊이 모델)도 함께 돌립니다.

- `A1 − A0` = 깊이 증가 + shortcut의 효과
- `A1 − A1(no-skip)` = **shortcut만의 효과**

**실행 상태:** Colab 실행 전 실험 설계본입니다.

**중요:** 모델 선택은 피험자 단위 validation으로 하고, 공식 test는 최종 모델을 정한 뒤 한 번만 확인합니다.


# 실험 A1: ResNet 1D 단독 효과 (검증셋으로 판단, 테스트는 마지막에 한 번)

| 셀 | 내용 | 언제 실행? |
|---|---|---|
| **0~12** | 학습 + **검증셋** 평가 | `USE_RESIDUAL=True` 로 SEED 0, 1, 2 → `False` 로 SEED 0, 1, 2 (총 6번) |
| **13 (🔒)** | **최종 테스트** (잠금: `FINAL_TEST=False`) | 4주차 비교가 끝나고 최종 모델을 정한 뒤 **한 번만** |

> ⚠️ "모두 실행"을 누르면 마지막 셀은 잠금 메시지(AssertionError)를 내고 멈춥니다. 정상 동작입니다.


## 0. 라이브러리와 실험 설정

필요한 패키지를 불러오고 이 실험의 변경점과 실행 방법을 코드 주석에서 확인합니다.



In [ ]:
# ============================================================
# 실험 A1: ResNet 1D 단독 효과 (A0 의 Conv 두 층 → residual block 두 개)
# ------------------------------------------------------------
# 기반: 4week/05_UCI_HAR_A0_Baseline_CNN.ipynb (헤드·데이터·학습·평가 동일)
#
# 목적: 5가지 기법 중 ResNet(shortcut + Add)만 넣었을 때의 효과 측정
#
# 변경 요약 (Ctrl+F 로 "[A1 변경]" 검색)
#   1) USE_RESIDUAL 플래그 추가 (True: ResNet, False: shortcut 만 뺀 대조군)
#   2) residual_block_1d 정의 (00 개념 노트북 1-4 와 같은 구조)
#   3) conv1(32)·conv2(64) → residual block 32·64 로 교체 (뒤의 MaxPooling 은 유지)
#   4) SUMMARY·그래프·결과표 이름을 A1 로 변경
#
# 실행 방법: USE_RESIDUAL=True 로 SEED 0 → 1 → 2, 이어서 False 로 SEED 0 → 1 → 2
#           SUMMARY 기록. 마지막 🔒 셀은 아직 실행하지 않음
# ============================================================

import os
import time   # [A0 변경] 학습·추론 시간 측정
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Dense
from tensorflow.keras.layers import GlobalAveragePooling1D       # [A0 변경] Flatten 대신 전역 평균 풀링
from tensorflow.keras.layers import Add, Activation               # [A1 변경] residual block 의 shortcut 더하기·ReLU
from tensorflow.keras.layers import BatchNormalization, Dropout   # [E1a 변경] 규제용 층
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping             # [E1a 변경] 조기 종료

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)




## 1. Seed

난수 시드를 고정합니다. 시드를 바꿔 재실행하면 초기화에 따른 변동을 볼 수 있습니다.



In [ ]:
# ============================================================
# 1. Seed
# ============================================================

# [E0 변경] 0, 1, 2 로 바꿔 3번 실행 (한 번의 결과는 우연일 수 있음)
SEED = 0

# [A1 변경] shortcut 사용 여부
# True : ResNet (Conv 두 개 결과 + 입력 shortcut 을 Add)
# False: 같은 Conv 두 개만 쌓고 shortcut 은 뺀 대조군
# 이유: residual block 은 블록마다 Conv 가 2개라 A0 보다 깊어짐
#       A0 와만 비교하면 "깊어져서 좋아진 것"과 "shortcut 덕분"을 구분할 수 없음
#       → 같은 깊이에서 shortcut 만 끈 모델과 비교해 shortcut 효과만 분리
USE_RESIDUAL = True

np.random.seed(SEED)
tf.random.set_seed(SEED)

# [A0 변경] B1·B2 와 같은 방식으로 시드 고정
# 이유: 같은 seed 끼리 A0 와 B1·B2 를 비교하므로 난수 설정도 똑같이 맞춤
#       set_random_seed 는 Python·NumPy·TF 난수를 한 번에 고정하고,
#       enable_op_determinism 은 GPU 연산 순서에 따른 작은 차이까지 줄임
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass




## 2. Data path

Colab에서 Drive를 연결하고 UCI HAR 데이터의 위치를 확인합니다.



In [ ]:
# ============================================================
# 2. Data path
# ============================================================

# [E0 변경] Drive 마운트 + 프로젝트 데이터 경로
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/가천대학교/2026-2/1. 기계학습프로그래밍/2week/DATA/UCI-HAR Dataset"

assert os.path.exists(DATA_PATH), f"데이터 경로를 확인하세요: {DATA_PATH}"




## 3. Signal names

입력으로 사용할 가속도계·자이로스코프 9개 채널의 파일명을 정의합니다.



In [ ]:
# ============================================================
# 3. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]




## 4. Load dataset

공식 train/test 분할의 센서 창, 행동 라벨, 피험자 번호를 읽습니다.



In [ ]:
# ============================================================
# 4. Load dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"

    y = np.loadtxt(y_path).astype(int)

    # label 1~6 -> 0~5
    y = y - 1

    # [E0 변경] 피험자 번호(1~30) 로드
    subject_path = f"{DATA_PATH}/{split}/subject_{split}.txt"

    subject = np.loadtxt(subject_path).astype(int)

    return X, y, subject


X_train, y_train, s_train = load_data("train")
X_test, y_test, s_test = load_data("test")




## 4-1. Subject-wise validation split

피험자 5·11·17·23·29를 검증용으로 떼어 학습 피험자와 겹치지 않게 합니다.



In [ ]:
# ============================================================
# 4-1. Subject-wise validation split
# ============================================================

# [E0 변경] 피험자 5명을 통째로 검증셋으로 분리 (같은 사람이 학습·검증에 걸치지 않게)
VAL_SUBJECTS = [5, 11, 17, 23, 29]

val_mask = np.isin(s_train, VAL_SUBJECTS)

X_val, y_val = X_train[val_mask], y_train[val_mask]

# [S0 변경] 검증 윈도우의 피험자 번호 보관
# 이유: 검증 피험자별(특히 5번, 17번) 정확도를 계산하기 위함
s_val = s_train[val_mask]

X_train, y_train = X_train[~val_mask], y_train[~val_mask]


print("\nDataset shape")
print("X_train:", X_train.shape, "(16 subjects)")
print("X_val  :", X_val.shape, "subjects:", VAL_SUBJECTS)
print("X_test :", X_test.shape, "(9 subjects, 마지막 🔒 셀에서만 사용)")




## 5. Normalization

학습 데이터의 평균·표준편차만 사용해 검증·테스트 입력을 변환합니다.



In [ ]:
# ============================================================
# 5. Normalization
# ============================================================

# [E0 변경] 평균/표준편차는 학습 16명으로만 계산
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = (X_train - mean) / (std + 1e-8)
X_val = (X_val - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)




## 6. One-hot encoding

정수 행동 라벨을 6개 범주의 원핫 벡터로 바꿉니다.



In [ ]:
# ============================================================
# 6. One-hot encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_val_onehot = to_categorical(
    y_val,
    num_classes=n_outputs
)




## 7. CNN Model (A1: residual block 두 개 + GAP 헤드)

A0의 Conv 두 층 자리에 residual block을 넣었습니다. 블록 뒤 MaxPooling과 GAP 헤드는 A0와 같습니다.

```text
입력 x ─────────────── (채널 다르면 1×1 Conv + BN) ──┐
  └→ Conv k3 → BN → ReLU → Conv k3 → BN ────────────┴ Add → ReLU
```


In [ ]:
# ============================================================
# 7. CNN Model (A1: residual block 두 개 + GAP 헤드)
# ============================================================

# [A1 변경] residual block 정의 (00 개념 노트북 1-4 와 같은 구조)
# 구조: Conv(k3) → BN → ReLU → Conv(k3) → BN  +  shortcut  → Add → ReLU
# 이유: Conv 결과 F(x) 에 입력 x 를 더해(y = F(x) + x), 블록이 "입력에서 바뀌는 부분"만 배우게 함
#       shortcut 은 정보와 gradient 가 그대로 지나가는 직통 경로가 됨
# padding="same": Add 하려면 두 입력의 시간 길이가 같아야 하므로 길이를 유지
#   (A0 는 padding 없음(valid)이라 Conv 마다 길이가 2씩 줄었음. 128→64→32 로 조금 달라짐)
def residual_block_1d(x, filters, use_residual=True, name="res"):

    shortcut = x

    y = Conv1D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv1")(x)
    y = BatchNormalization(name=f"{name}_bn1")(y)
    y = Activation("relu", name=f"{name}_relu1")(y)

    y = Conv1D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv2")(y)
    y = BatchNormalization(name=f"{name}_bn2")(y)

    if use_residual:

        # 입력 채널 수(예: 9)와 출력 채널 수(32)가 다르면 Add 할 수 없으므로
        # 1×1 Conv 로 shortcut 채널 수를 맞춤 (projection shortcut)
        if x.shape[-1] != filters:
            shortcut = Conv1D(filters, 1, padding="same", use_bias=False, name=f"{name}_projection")(shortcut)
            shortcut = BatchNormalization(name=f"{name}_projection_bn")(shortcut)

        y = Add(name=f"{name}_add")([shortcut, y])

    # use_residual=False 이면 Add 없이 같은 Conv 두 개만 거친 결과
    return Activation("relu", name=f"{name}_out")(y)


inputs = Input(shape=(128, 9))


# [A1 변경] conv1(32, k3) + BN  →  residual block 32
res1 = residual_block_1d(inputs, 32, use_residual=USE_RESIDUAL, name="res32")

pool1 = MaxPooling1D(
    pool_size=2
)(res1)


# [A1 변경] conv2(64, k3) + BN  →  residual block 64
res2 = residual_block_1d(pool1, 64, use_residual=USE_RESIDUAL, name="res64")

pool2 = MaxPooling1D(
    pool_size=2
)(res2)


# [A0 변경] Flatten → Dropout → Dense(128) → Dropout  ⇒  GlobalAveragePooling1D → Dropout(0.4)
# 이유: B1·B2 의 헤드와 똑같이 맞춰, 4주차 비교에서 "블록 구조" 차이만 남기기 위함
#       GAP 는 64 개 필터마다 32 개 시점(same padding)을 평균 내서 64 개 값으로 요약함
#       → Flatten(32×64=2,048 개 값) + Dense(128) 보다 파라미터가 크게 줄어듦
#       Dropout 0.4 와 Dense(128) 제거도 B1·B2 헤드와 같게 하기 위함
# 참고: 2주차 E1b 도 GAP 를 썼지만 Dense(128)·Dropout 0.5 를 유지했으므로 A0 와는 다름
gap = GlobalAveragePooling1D()(pool2)

gap = Dropout(0.4)(gap)


outputs = Dense(
    n_outputs,
    activation="softmax"
)(gap)




## 8. Model

모델을 만들고 손실 함수, 옵티마이저, 평가 지표를 설정합니다.



In [ ]:
# ============================================================
# 8. Model
# ============================================================

model = Model(
    inputs=inputs,
    outputs=outputs
)


model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


model.summary()




## 9. Train

학습 피험자의 데이터로 모델을 훈련하고 학습 이력을 저장합니다.



In [ ]:
# ============================================================
# 9. Train
# ============================================================

# [E1a 변경] val_loss 가 10 에폭 연속 나아지지 않으면 멈추고 최적 가중치로 복원
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

# [A0 변경] 학습 시간 측정 (B1·B2 와 비교)
train_start = time.perf_counter()

history = model.fit(
    X_train,
    y_train_onehot,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_onehot),
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

training_seconds = time.perf_counter() - train_start

print(f"training time: {training_seconds:.1f} sec")


## 10. Validation evaluation   [S0 추가]

검증 피험자에 예측을 적용해 정확도·Macro F1과 혼동 행렬을 계산합니다.



In [ ]:
# ============================================================
# 10. Validation evaluation   [S0 추가]
# ============================================================

# [S0 변경] test 대신 검증셋으로 예측하고 평가
# 이유: 방법 비교·선택은 검증셋으로만 함. test 는 마지막 🔒 셀에서 한 번만
y_val_pred = np.argmax(
    model.predict(X_val, verbose=0),
    axis=1
)

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

val_accuracy = accuracy_score(y_val, y_val_pred)

val_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)

cm_val = confusion_matrix(y_val, y_val_pred, labels=range(n_outputs))


print("\n==============================")
print("VALIDATION RESULTS (subjects 5, 11, 17, 23, 29)")
print("==============================")

print(f"Accuracy  : {val_accuracy:.4f}")
print(f"F1-score  : {val_f1:.4f}")

print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

print("CONFUSION MATRIX (validation)")
print(cm_val)




## 11. SIT/STAND focus on validation subjects   [S0 추가]

검증 피험자별 SIT/STAND 정확도를 보고 5·17번 핵심 지표와 나머지 지표를 분리합니다.



In [ ]:
# ============================================================
# 11. SIT/STAND focus on validation subjects   [S0 추가]
# ============================================================

# [S0 변경] 검증 피험자별 전체 정확도 + SIT/STAND 정확도
# SIT/STAND 정확도 = 실제로 앉음·섬인 윈도우 중에서 정확히 맞힌 비율 (6개 클래스 예측 기준)
# 이유: 전체 정확도에는 걷기·눕기가 섞여 있어 SIT/STAND 개선이 묻힘 → 문제 클래스만 따로 측정
SIT, STAND = 3, 4

FOCUS_SUBJECTS = [5, 17]   # 기울기 차이가 작은 검증 피험자 (5번 12.1°, 17번 14.9°)

print("\n==============================")
print("PER-SUBJECT (validation)")
print("==============================")
print("subject | all acc | SIT/STAND acc | SIT->STAND | STAND->SIT")

for s in VAL_SUBJECTS:

    m = (s_val == s)

    ms = m & np.isin(y_val, [SIT, STAND])

    sit_to_stand = np.sum(m & (y_val == SIT) & (y_val_pred == STAND))

    stand_to_sit = np.sum(m & (y_val == STAND) & (y_val_pred == SIT))

    mark = "   <- 핵심" if s in FOCUS_SUBJECTS else ""

    print(
        f"{s:7d} | {accuracy_score(y_val[m], y_val_pred[m]):.4f}  | "
        f"{accuracy_score(y_val[ms], y_val_pred[ms]):.4f}        | "
        f"{sit_to_stand:10d} | {stand_to_sit:10d}{mark}"
    )

# 핵심 지표: 5번 + 17번의 SIT/STAND 윈도우를 합쳐서 계산한 정확도
mf = np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

focus_ss_acc = accuracy_score(y_val[mf], y_val_pred[mf])

# 보조 지표: 나머지 검증 피험자(11, 23, 29)의 SIT/STAND 정확도
mo = ~np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

other_ss_acc = accuracy_score(y_val[mo], y_val_pred[mo])

val_ss_err = cm_val[SIT, STAND] + cm_val[STAND, SIT]

val_dynamic_err = cm_val[0:3, 0:3].sum() - np.trace(cm_val[0:3, 0:3])

best_epoch = int(np.argmin(history.history["val_loss"])) + 1


# [A0 변경] 샘플당 추론 시간 측정 (B1·B2 와 같은 방법)
# 처음 몇 번은 그래프 준비 시간이 섞이므로 warm-up 3번 후 20번 반복해 평균
# GPU 종류에 따라 절대값이 달라지므로 같은 Colab 런타임에서 잰 값끼리만 비교
timing_batch = tf.convert_to_tensor(X_val[:256].astype("float32"))

for _ in range(3):
    model(timing_batch, training=False).numpy()

repeats = 20

inference_start = time.perf_counter()

for _ in range(repeats):
    model(timing_batch, training=False).numpy()

inference_seconds = time.perf_counter() - inference_start

inference_ms_per_sample = inference_seconds * 1000 / (repeats * len(timing_batch))


# [A0 변경] 학습·추론 시간 추가 (B1·B2 SUMMARY 와 같은 항목)
# [A1 변경] SUMMARY 이름을 A1-residual / A1-no-skip 으로 구분
EXPERIMENT_NAME = "A1-residual" if USE_RESIDUAL else "A1-no-skip"

print("\n==============================")
print("SUMMARY (결과표에 기록)")
print("==============================")

print(
    f"{EXPERIMENT_NAME} | seed {SEED} | val acc {val_accuracy:.4f} | val F1 {val_f1:.4f} | "
    f"focus SIT/STAND(5,17) {focus_ss_acc:.4f} | other SIT/STAND(11,23,29) {other_ss_acc:.4f} | "
    f"val SIT<->STAND {val_ss_err} ({cm_val[SIT, STAND]}+{cm_val[STAND, SIT]}) | val dynamic {val_dynamic_err} | "
    f"epochs {len(history.history['val_loss'])} (best {best_epoch}) | params {model.count_params():,} | "
    f"train sec {training_seconds:.1f} | inference ms/sample {inference_ms_per_sample:.4f}"
)




## 12. Learning curve

학습 곡선을 보고 손실과 검증 지표가 에폭에 따라 어떻게 변하는지 살펴봅니다.



In [ ]:
# ============================================================
# 12. Learning curve
# ============================================================

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("epoch")
plt.title(f"{EXPERIMENT_NAME} loss (seed {SEED})")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.xlabel("epoch")
plt.title(f"{EXPERIMENT_NAME} accuracy (seed {SEED})")
plt.legend()

plt.tight_layout()
plt.show()

## 결과 기록표

A0 결과와 같은 seed끼리 비교합니다.

| 실험 | seed | Val Accuracy | Macro F1 | Focus SIT/STAND (5,17) | Other SIT/STAND (11,23,29) | SIT↔STAND | Params | Best epoch | Train sec | ms/sample |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| A1-residual | 0 |  |  |  |  |  |  |  |  |  |
| A1-residual | 1 |  |  |  |  |  |  |  |  |  |
| A1-residual | 2 |  |  |  |  |  |  |  |  |  |
| A1-no-skip | 0 |  |  |  |  |  |  |  |  |  |
| A1-no-skip | 1 |  |  |  |  |  |  |  |  |  |
| A1-no-skip | 2 |  |  |  |  |  |  |  |  |  |

### 해석 기준

- `A1-residual − A0`: ResNet 블록을 넣은 전체 효과(깊이 + shortcut)입니다.
- `A1-residual − A1-no-skip`: 같은 깊이에서 **shortcut만의 효과**입니다.
- 모델이 얕아서(Conv 4층) shortcut 효과는 작을 수 있습니다. 차이가 없으면 "이 깊이에서는 shortcut이 필요 없다"는 결과입니다.
- 세 seed의 평균·표준편차와 같은 seed끼리의 차이 방향이 반복되는지 함께 봅니다.


## 🔒 최종 테스트 (한 번만 실행)
4주차 후보(A0~A5, B1~B4)를 **검증 지표**로 비교해 최종 모델을 정한 뒤에만 실행합니다.
실행 순서: 위 셀들을 원하는 SEED 로 실행 → 아래 셀에서 `FINAL_TEST = True` 로 바꿔 실행


In [ ]:
# ============================================================
# 13. FINAL TEST   [S0 추가]
# ============================================================

# [S0 변경] 테스트 평가를 별도 셀로 분리하고 잠금 (A1 도 그대로 유지)
# 이유: test 점수를 보고 방법을 고르면 test 에 맞춘 선택이 되어 최종 점수의 의미가 사라짐.
#       검증셋으로 최종 모델을 정한 뒤에만 True 로 바꿔 실행 (위에서 학습한 model 을 그대로 평가)
FINAL_TEST = False

assert FINAL_TEST, (
    "테스트셋은 최종 모델을 정한 뒤 한 번만 봅니다. "
    "지금은 위 셀들의 검증 결과로 비교하세요. (최종 평가 때 FINAL_TEST = True)"
)

y_pred = np.argmax(
    model.predict(X_test, verbose=0),
    axis=1
)

accuracy = accuracy_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

cm = confusion_matrix(y_test, y_pred, labels=range(n_outputs))


print("\n==============================")
print("FINAL TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"F1-score  : {f1:.4f}")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

print("CONFUSION MATRIX (test)")
print(cm)


print("\n==============================")
print("PER-SUBJECT ACCURACY (test)")
print("==============================")

subject_acc = {}

for s in np.unique(s_test):

    m = (s_test == s)

    subject_acc[s] = accuracy_score(y_test[m], y_pred[m])

    print(f"subject {s:2d}: {subject_acc[s]:.4f}")

worst_subject = min(subject_acc, key=subject_acc.get)

sit_stand_err = cm[SIT, STAND] + cm[STAND, SIT]

dynamic_err = cm[0:3, 0:3].sum() - np.trace(cm[0:3, 0:3])

print(
    f"\n{EXPERIMENT_NAME} TEST | seed {SEED} | acc {accuracy:.4f} | macro F1 {f1:.4f} | "
    f"SIT<->STAND {sit_stand_err} | dynamic {dynamic_err} | "
    f"worst subject {worst_subject} ({subject_acc[worst_subject]:.4f})"
)